# Fraud Detection System
## 04 — Modeling

| | |
|---|---|
| **Author** | Jose Lopez Pino |
| **Framework** | CRISP-DM + Lean |
| **Phase** | 4 — Modeling |
| **Project** | fraud-detection · applied-data-science-portfolio |
| **Dataset** | Credit Card Fraud Detection — ULB Machine Learning Group (Kaggle) |
| **Date** | 2026-04 |

---

> **Executive Summary:**
> This notebook trains and compares 9 model configurations (3 models × 3 imbalance strategies)
> using AUC-PR as the primary metric. An Isolation Forest is included as an unsupervised benchmark
> to demonstrate paradigm judgment. The winning configuration — best model + best imbalance strategy
> — is selected here and passed to notebook 05 for threshold calibration and final evaluation.
> The test set remains locked throughout this notebook.

## Table of Contents

1. [Load Processed Data](#1-load-processed-data)
2. [Modeling Strategy](#2-modeling-strategy)
3. [Baseline — Logistic Regression](#3-baseline--logistic-regression)
4. [Random Forest](#4-random-forest)
5. [XGBoost](#5-xgboost)
6. [Unsupervised Benchmark — Isolation Forest](#6-unsupervised-benchmark--isolation-forest)
7. [Model Comparison — AUC-PR Leaderboard](#7-model-comparison--aucpr-leaderboard)
8. [Winner Selection](#8-winner-selection)
9. [LEAN Filter — Waste Elimination Review](#9-lean-filter--waste-elimination-review)
10. [Decisions Log](#10-decisions-log)
11. [Next Steps — Notebook 05 Preview](#11-next-steps--notebook-05-preview)

In [ ]:
# ===== Environment Setup =====
import warnings
warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')

# === Standard library ===
from pathlib import Path

# === Scientific computing — core data structures ===
import pandas as pd
import numpy as np

# === Visualization — base plotting ===
import matplotlib.pyplot as plt

# === Visualization — statistical plotting ===
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PROCESSED  = Path('../data/processed')
REPORTS_FIGURES = Path('../reports/figures')
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)

print('Environment ready.')
print(f'Processed data path: {DATA_PROCESSED}')

---
## 1. Load Processed Data

In [ ]:
# ===== Load all splits from notebook 03 =====
X_train       = pd.read_csv(DATA_PROCESSED / 'X_train.csv')
y_train       = pd.read_csv(DATA_PROCESSED / 'y_train.csv').squeeze()
X_train_smote = pd.read_csv(DATA_PROCESSED / 'X_train_smote.csv')
y_train_smote = pd.read_csv(DATA_PROCESSED / 'y_train_smote.csv').squeeze()
X_val         = pd.read_csv(DATA_PROCESSED / 'X_val.csv')
y_val         = pd.read_csv(DATA_PROCESSED / 'y_val.csv').squeeze()

# Test set loaded but NOT used in this notebook
X_test = pd.read_csv(DATA_PROCESSED / 'X_test.csv')
y_test = pd.read_csv(DATA_PROCESSED / 'y_test.csv').squeeze()

print('=== Data loaded ===')
print(f'X_train       : {X_train.shape} | fraud rate: {y_train.mean()*100:.4f}%')
print(f'X_train_smote : {X_train_smote.shape} | fraud rate: {y_train_smote.mean()*100:.4f}%')
print(f'X_val         : {X_val.shape} | fraud rate: {y_val.mean()*100:.4f}%')
print(f'X_test        : {X_test.shape} | LOCKED — not used until nb05')

---
## 2. Modeling Strategy

### 2.1 Evaluation metric — why AUC-PR

All models are evaluated on the **validation set** using **AUC-PR** (Area Under
the Precision-Recall Curve) as the primary metric.

AUC-PR is correct here because:
- With 0.17% fraud rate, AUC-ROC is optimistic and misleading
- AUC-PR focuses on the minority class (fraud) — the class that matters
- A random classifier has AUC-PR ≈ fraud rate (0.0017) — any real model must beat this

### 2.2 Experiment matrix

| Model | No balancing | class_weight | SMOTE |
|---|:---:|:---:|:---:|
| Logistic Regression | ✅ | ✅ | ✅ |
| Random Forest | ✅ | ✅ | ✅ |
| XGBoost | ✅ | ✅ | ✅ |
| **Isolation Forest** | ✅ (unsupervised benchmark) | — | — |

### 2.3 Helper function

In [ ]:
# === Model evaluation ===
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    roc_auc_score, classification_report
)

# ===== Helper: evaluate model on validation set =====
def evaluate_model(model, X_val, y_val, model_name):
    """
    Evaluate a fitted classifier on the validation set.
    Returns AUC-PR, AUC-ROC, and prints a summary.
    Primary metric: AUC-PR (correct for imbalanced fraud detection).
    """
    y_prob = model.predict_proba(X_val)[:, 1]
    y_pred = model.predict(X_val)

    auc_pr  = average_precision_score(y_val, y_prob)
    auc_roc = roc_auc_score(y_val, y_prob)

    print(f'\n=== {model_name} ===')
    print(f'AUC-PR  (primary) : {auc_pr:.4f}')
    print(f'AUC-ROC           : {auc_roc:.4f}')
    print(classification_report(y_val, y_pred, target_names=['Legitimate', 'Fraud']))

    return {'model': model_name, 'auc_pr': auc_pr, 'auc_roc': auc_roc}

# Storage for leaderboard
results = []
print('Helper function ready.')

---
## 3. Baseline — Logistic Regression

In [ ]:
# === Logistic Regression ===
from sklearn.linear_model import LogisticRegression

# ===== LR — No balancing =====
lr_raw = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_raw.fit(X_train, y_train)
results.append(evaluate_model(lr_raw, X_val, y_val, 'LR — No balancing'))

In [ ]:
# ===== LR — class_weight balanced =====
lr_cw = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')
lr_cw.fit(X_train, y_train)
results.append(evaluate_model(lr_cw, X_val, y_val, 'LR — class_weight'))

In [ ]:
# ===== LR — SMOTE =====
lr_smote = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_smote.fit(X_train_smote, y_train_smote)
results.append(evaluate_model(lr_smote, X_val, y_val, 'LR — SMOTE'))

---
## 4. Random Forest

In [ ]:
# === Random Forest ===
from sklearn.ensemble import RandomForestClassifier

# ===== RF — No balancing =====
rf_raw = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_raw.fit(X_train, y_train)
results.append(evaluate_model(rf_raw, X_val, y_val, 'RF — No balancing'))

In [ ]:
# ===== RF — class_weight balanced =====
rf_cw = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                class_weight='balanced', n_jobs=-1)
rf_cw.fit(X_train, y_train)
results.append(evaluate_model(rf_cw, X_val, y_val, 'RF — class_weight'))

In [ ]:
# ===== RF — SMOTE =====
rf_smote = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_smote.fit(X_train_smote, y_train_smote)
results.append(evaluate_model(rf_smote, X_val, y_val, 'RF — SMOTE'))

---
## 5. XGBoost

In [ ]:
# === XGBoost ===
from xgboost import XGBClassifier

# Scale positive weight for no-balancing version
# Ratio: n_legitimate / n_fraud in training set
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# ===== XGB — No balancing =====
xgb_raw = XGBClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    eval_metric='aucpr',
    verbosity=0
)
xgb_raw.fit(X_train, y_train)
results.append(evaluate_model(xgb_raw, X_val, y_val, 'XGB — No balancing'))

In [ ]:
# ===== XGB — scale_pos_weight (equivalent to class_weight) =====
xgb_cw = XGBClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    verbosity=0
)
xgb_cw.fit(X_train, y_train)
results.append(evaluate_model(xgb_cw, X_val, y_val, 'XGB — scale_pos_weight'))

In [ ]:
# ===== XGB — SMOTE =====
xgb_smote = XGBClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    eval_metric='aucpr',
    verbosity=0
)
xgb_smote.fit(X_train_smote, y_train_smote)
results.append(evaluate_model(xgb_smote, X_val, y_val, 'XGB — SMOTE'))

---
## 6. Unsupervised Benchmark — Isolation Forest

Isolation Forest does not use labels during training — it detects anomalies
based on how easy it is to isolate a point from the rest of the data.

This benchmark answers: **is labeled data necessary for this problem?**
If Isolation Forest matches supervised models, labels add little value.
If supervised models win clearly, labels justify the annotation cost.

In [ ]:
# === Isolation Forest ===
from sklearn.ensemble import IsolationForest

# ===== Isolation Forest — unsupervised anomaly detection =====
# contamination = known fraud rate in training set
contamination = y_train.mean()

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=contamination,
    random_state=RANDOM_STATE
)
iso_forest.fit(X_train)  # no labels used

# Convert anomaly scores to fraud probability proxy
# IsolationForest: -1 = anomaly (fraud), 1 = normal (legitimate)
# score_samples returns negative anomaly scores — lower = more anomalous
iso_scores = -iso_forest.score_samples(X_val)  # flip sign: higher = more anomalous
iso_preds  = iso_forest.predict(X_val)
iso_preds_binary = np.where(iso_preds == -1, 1, 0)  # -1 → fraud (1)

iso_auc_pr  = average_precision_score(y_val, iso_scores)
iso_auc_roc = roc_auc_score(y_val, iso_scores)

print('=== Isolation Forest — Unsupervised Benchmark ===')
print(f'AUC-PR  (primary) : {iso_auc_pr:.4f}')
print(f'AUC-ROC           : {iso_auc_roc:.4f}')
print(classification_report(y_val, iso_preds_binary, target_names=['Legitimate', 'Fraud']))

results.append({'model': 'Isolation Forest (unsupervised)', 'auc_pr': iso_auc_pr, 'auc_roc': iso_auc_roc})

---
## 7. Model Comparison — AUC-PR Leaderboard

In [ ]:
# ===== AUC-PR Leaderboard =====
leaderboard = pd.DataFrame(results).sort_values('auc_pr', ascending=False).reset_index(drop=True)
leaderboard.index += 1
leaderboard.columns = ['Model', 'AUC-PR (primary)', 'AUC-ROC']

print('=== Model Leaderboard — sorted by AUC-PR ===')
print(leaderboard.to_string())

In [ ]:
# ===== Leaderboard visualization =====
fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#1565C0' if i == 0 else '#90CAF9' for i in range(len(leaderboard))]
bars = ax.barh(leaderboard['Model'][::-1], leaderboard['AUC-PR (primary)'][::-1],
               color=colors[::-1], edgecolor='white')

for bar, val in zip(bars, leaderboard['AUC-PR (primary)'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('AUC-PR (higher = better)')
ax.set_title('Model Comparison — AUC-PR Leaderboard\n(Primary metric for imbalanced fraud detection)',
             fontweight='bold')
ax.axvline(y_val.mean(), color='red', linestyle='--', alpha=0.7,
           label=f'Random classifier baseline (AUC-PR ≈ {y_val.mean():.4f})')
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / 'model_leaderboard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== Precision-Recall curves — top 3 models =====
top3_models = {
    leaderboard.iloc[0]['Model']: None,
    leaderboard.iloc[1]['Model']: None,
    leaderboard.iloc[2]['Model']: None,
}

# Map model names to fitted objects
model_objects = {
    'LR — No balancing': lr_raw,
    'LR — class_weight': lr_cw,
    'LR — SMOTE': lr_smote,
    'RF — No balancing': rf_raw,
    'RF — class_weight': rf_cw,
    'RF — SMOTE': rf_smote,
    'XGB — No balancing': xgb_raw,
    'XGB — scale_pos_weight': xgb_cw,
    'XGB — SMOTE': xgb_smote,
}

fig, ax = plt.subplots(figsize=(9, 6))
colors_pr = ['#1565C0', '#E53935', '#2E7D32']

for (name, _), color in zip(top3_models.items(), colors_pr):
    if name in model_objects:
        y_prob = model_objects[name].predict_proba(X_val)[:, 1]
        precision, recall, _ = precision_recall_curve(y_val, y_prob)
        auc_pr = average_precision_score(y_val, y_prob)
        ax.plot(recall, precision, label=f'{name} (AUC-PR={auc_pr:.4f})', color=color)

ax.axhline(y_val.mean(), color='gray', linestyle='--', alpha=0.7, label='Random baseline')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — Top 3 Models', fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / 'pr_curves_top3.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Winner Selection

In [ ]:
# ===== Select winning model =====
winner_name = leaderboard.iloc[0]['Model']
winner_auc_pr = leaderboard.iloc[0]['AUC-PR (primary)']

# Map winner to fitted object
winner_model = model_objects.get(winner_name)

print('=== Winner Selection ===')
print(f'Winner          : {winner_name}')
print(f'AUC-PR (val)    : {winner_auc_pr:.4f}')
print()
print('This model goes to notebook 05 for:')
print('  - Threshold calibration via Expected Loss function')
print('  - Final evaluation on LOCKED test set')
print('  - Comparison against naive baselines')

In [ ]:
# ===== Save winner model =====
import joblib
from pathlib import Path

models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

model_path = models_dir / 'model_v1_2026-04.pkl'
joblib.dump(winner_model, model_path)

print(f'Model saved: {model_path}')
print(f'Size: {model_path.stat().st_size / 1024:.1f} KB')

---
## 9. LEAN Filter — Waste Elimination Review

| LEAN Question | Answer | Action |
|---|---|---|
| Was hyperparameter tuning done here? | ✅ No (Lean applied) — default params first, tune only winner in nb05 | Proceed |
| Was the test set used? | ✅ No — locked until notebook 05 | Proceed |
| Does AUC-PR drive all decisions? | ✅ Yes — leaderboard sorted by AUC-PR, not accuracy or AUC-ROC | Proceed |
| Is the Isolation Forest comparison valuable? | ✅ Yes — demonstrates supervised vs unsupervised paradigm judgment | Proceed |
| Were all 9 configurations necessary? | ✅ Yes — imbalance strategy comparison is a key modeling decision | Proceed |

---
## 10. Decisions Log

| # | Decision | Rationale | Alternatives Considered | LEAN Value? |
|---|---|---|---|---|
| 1 | Use AUC-PR as primary ranking metric | Correct metric for imbalanced data — AUC-ROC is optimistic with 0.17% fraud rate | AUC-ROC, F1 | ✅ |
| 2 | Default hyperparameters for all models | Lean principle — establish baseline before tuning; avoid premature optimization | Grid search in nb04 | ✅ |
| 3 | n_estimators=100 for RF and XGBoost | Sufficient for comparison — tuning reserved for winner in nb05 | 50, 200, 500 | ✅ |
| 4 | Include Isolation Forest as benchmark | Demonstrates paradigm judgment — answers whether labeled data is necessary | Skip unsupervised | ✅ |
| 5 | Save winner with joblib, not pickle | joblib is more efficient for numpy arrays — sklearn standard | pickle | ✅ |
| 6 | XGBoost uses scale_pos_weight instead of class_weight | XGBoost does not support sklearn class_weight API — scale_pos_weight is the native equivalent | SMOTE only for XGB | ✅ |

---
## 11. Next Steps — Notebook 05 Preview

**Notebook 05 — Evaluation** will:

- Load the winner model from `models/model_v1_2026-04.pkl`
- Calibrate the classification threshold using the Expected Loss function from notebook 01
- Find `t*` = threshold that minimizes `FN × cost_fn + FP × cost_fp`
- Run final evaluation on the **locked test set** at threshold `t*`
- Compare Expected Loss vs naive baselines (all legitimate, all fraud)
- Produce the final metrics table: Recall, Precision, F1, AUC-PR, Expected Loss

**Lean filter for nb05:** One threshold optimization method — cost function.
No Youden index, no F1-optimal threshold — the business cost function is the arbiter.

---

**← Previous:** [03 — Data Preparation](./03_data_preparation.ipynb)
**Next →** [05 — Evaluation](./05_evaluation.ipynb)